In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
from glm_hmm_utils import *
from neurodatatypes import Session
import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:
DPATH = 'X:\Widefield'
MICE = ['mSM63','mSM64','mSM65','mSM66']
#MICE = ['Fez71','Fez72','Fez73','Fez74','Fez75']
#MICE = ['CSP22','CSP23','CSP32','CSP38']
sessions = []
mousenames = []

# all sessions together
for mouse in tqdm(MICE):
    sessionlist = Session.get_sessions(DPATH, mouse, max_nochoice = 5, modality = 2, min_trials = 200, discrim_min = .5, discrim_max = 1, assisted_cutoff = .95, singlespout_cutoff = .02) # This should be all audio discriminatin sessions
    sessions.extend(sessionlist)
    mousenames.extend([mouse]*len(sessionlist))
    print(f'There are {len(sessionlist)} sessions for {mouse}')
print(f'There are {len(sessions)} sessions total')

In [ ]:
#model = GlmHmm(sessions, 'actual_rate')
model = GlmHmm(sessions,
               'target_rate',
               input_terms_list=['coherence','bias'])

In [ ]:
#model.train_new_model('map', N_iters = 5000, N_states=3, alpha=2, sigma=.5) 
#model.train_n_models(10, mthd='map', N_iters = 5000, N_states=3, alpha=2, sigma=.5) 
#model.train_n_models(10, mthd='map', N_iters = 2000, N_states=3, alpha=2, sigma=.75) # Fez 3 states
#model.train_n_models(10, mthd='map', N_iters = 2000, N_states=4, alpha=2, sigma=.75)  # Fez 4 states

#model.train_n_models(10, mthd='map', N_iters = 2000, N_states=3, alpha=2, sigma=.75) # CSP testing
#model.train_n_models(10, mthd='map', N_iters = 2000, N_states=3, alpha=1, sigma=.25) # CSP 3 state
model.train_n_models(10, mthd='map', N_iters = 2000, N_states=4, alpha=2, sigma=.5) # CSP 4 state

In [ ]:
#del model.label_inds
_, best_mdl_index, _ = model.return_best_model('ll', model.inpts, model.outputs, train_outputs=None, silent=False)

In [ ]:
# Plotting
cols = ['black','red','blue','orange','pink']
weights = model.return_ordered_weights(modelindex=best_mdl_index)
#weights = mdl.observations.params
weights = np.squeeze(weights)
for i in range(weights.shape[0]):
    plt.plot(weights[i,:], '-o',color=cols[i])

plt.xticks(range(len(model.input_terms_list)))
plt.gca().set_xticklabels(model.input_terms_list)
plt.axhline(y = 0, color = 'black', linestyle = '--')
plt.legend(['Engaged State','Left Bias State','Right Bias State'])
#plt.title(f'Weights for merged sessions: {n_states} states')
plt.ylabel('GLM Weight')

import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
#plt.savefig(r'C:\Data\churchland\state_manuscript_new_figs\raw_figures\glmhmm_fitting_supp\weights.pdf', format='pdf', dpi=1000, bbox_inches='tight')

In [ ]:
states = model.return_ordered_states(modelindex=best_mdl_index)
for isess,sess in enumerate(states):
    plt.figure(dpi=50)
    for i in range(weights.shape[0]):
        plt.plot(sess[:,i],color=cols[i])
    plt.title(mousenames[isess])
    plt.show()

In [ ]:
SESSION_IND = 0

plt.figure(dpi=500)
for i in range(weights.shape[1]):
    plt.plot(states[SESSION_IND][:,i],color=cols[i])
#plt.savefig(r'C:\Data\churchland\Manuscripts\state_tim_encoding_manuscript\state_posteriors.pdf', format='pdf', dpi=1000, bbox_inches='tight')

In [ ]:
model.save(r'X:\Widefield\glm_hmm_models')